# **The Crying Baby**
---

## **Partially Observable MDP**

Lauri et al. state that 
> Noisy sensing, imperfect control, and environment changes are defining characteristics of many real-world robot tasks. The partially observable Markov decision process (POMDP) provides a principled mathematical framework for modeling and solving robot decision and control tasks under uncertainty. A POMDP models a decision-maker, also called the agent, for a system with incomplete state information. At each time step, the agent executes an action that yields some reward depending on the current state and the action, and results in a stochastic transition to a new system state. The agent acquires information about the new state via noisy observations. A solution to a POMDP is a policy that prescribes the agent’s action conditioned on its past history of actions and observations [1].

It is formulated as follows:

$$
M = (S,A,T,R,O,Z,\gamma)
$$

where:

- $S$: State Space
- $A$: Action Space
- $T$: Transition Probabilities
- $R$: Reward Function
- $O$: Observation Space
- $Z$: Observation Model
- $\gamma$: Discount Factor

The agent receives an observation of the current state rather than the true state (potentially imperfect observations). Using past observations the agent builds a **belief** of their underlying state, which can be represented by a probability distribution over the states.

---

## **Crying Baby POMDP**

*The Crying Baby* is a simple decision-making under uncertainty problem. Where the decision-making agent is a caregiver. The baby can be in two states, either **hungry** or **full**. The caregiver however, doesn't know the true current state of the baby. All it sees, are imperfect observations of the environment which are the baby either **crying** or being **quiet**. Based on these observations, it can choose one of the three actions, to **feed** the baby, **ignore** or **sing** to the baby.

The ultimate goal of the caregiver is to keep the baby full with the lowest cost (higher reward). Reflecting on the fact that crying is a strong indicator that the baby is hungry or the baby being quiet means the baby is most likely full.

### **State Space**

The state represents the baby's actual condition, which cannot be observed directly.

$$
S = \{\text{Hungry}, \text{Full}\}
$$

In [2]:
from enum import Enum

In [3]:
class State(Enum):
    HUNGRY = 0
    FULL = 1

### **Action Space**

The caregiver can choose one of the three actions at each timestep.

$$
A = \{\text{Feed}, \text{Ignore}, \text{Sing}\}
$$

The `Sing` action illustrates an impoertant characteristic of POMDPS, Singing does not change the baby's underlying condition, but it can reveal information about that condition. This additional observation can help the agent form a more accurate estimate of the true state. 

In [4]:
class Action(Enum):
    IGNORE = 0
    SING = 1
    FEED = 2

### **Transition Probabilities**

The transition model describes how the baby's hidden state can change after
taking an action. It is represented by

$$
T(s' \mid s,a) = P(s_{t+1}=s' \mid s_t=s, a_t=a).
$$

Feeding the baby always results in the baby becoming Full.

$$
T(\text{Full} \mid \text{Hungry}, \text{Feed}) = 1
$$

$$
T(\text{Hungry} \mid \text{Hungry}, \text{Feed}) = 0
$$

$$
T(\text{Full} \mid \text{Full}, \text{Feed}) = 1
$$

$$
T(\text{Hungry} \mid \text{Full}, \text{Feed}) = 0
$$

If the baby is hungry and is not fed, it remains hungry.

$$
T(\text{Hungry} \mid \text{Hungry}, a) = 1
$$

$$
T(\text{Full} \mid \text{Hungry}, a) = 0
$$

If the baby is currently not hungry, there is a 10% chance that it becomes hungry during the next time step If the agent chooses either `Sing` or `Ignore`.

$$
T(\text{Hungry} \mid \text{Full}, a) = 0.1
$$

$$
T(\text{Full} \mid \text{Full}, a) = 0.9
$$

where $a \in \{\text{Sing}, \text{Ignore}\}$.

Notice that the transition probabilities are the same for `Ignore` and `Sing`.

$$
T(s' \mid s, \text{Ignore}) = T(s' \mid s, \text{Sing})
\\
s, s' \in S
$$

This indicates that ignoring the baby or singing to him has the same impact on the baby's true condition but these actions differ in terms of following observations and rewards which will be discussed later. 


In [5]:
def transition_probability(next_state: State, state: State, action: Action):

    if action == Action.FEED:
        return (1 if next_state == State.FULL else 0)

    if state == State.HUNGRY:
        return (1 if next_state == State.HUNGRY else 0)

    return (0.1 if next_state == State.HUNGRY else 0.9)

In [6]:
# Testing

print("Feeding:")

action = Action.FEED
print(f"T(Full|Hungry, Feed) = {transition_probability(State.FULL, State.HUNGRY, action)}")
print(f"T(Hungry|Hungry, Feed) = {transition_probability(State.HUNGRY, State.HUNGRY, action)}")
print(f"T(Full|Full, Feed) = {transition_probability(State.FULL, State.FULL, action)}")
print(f"T(Hungry|Full, Feed) = {transition_probability(State.HUNGRY, State.FULL, action)}")

print("\na is Singing or Ignoring:")
action = Action.IGNORE
print(f"T(Hungry|Hungry, a) = {transition_probability(State.HUNGRY, State.HUNGRY, action)}")
print(f"T(Full|Hungry, a) = {transition_probability(State.FULL, State.HUNGRY, action)}")
print(f"T(Hungry|Full, a) = {transition_probability(State.HUNGRY, State.FULL, action)}")
print(f"T(Full|Full, a) = {transition_probability(State.FULL, State.FULL, action)}")


Feeding:
T(Full|Hungry, Feed) = 1
T(Hungry|Hungry, Feed) = 0
T(Full|Full, Feed) = 1
T(Hungry|Full, Feed) = 0

a is Singing or Ignoring:
T(Hungry|Hungry, a) = 1
T(Full|Hungry, a) = 0
T(Hungry|Full, a) = 0.1
T(Full|Full, a) = 0.9


### **Reward Function**

The reward model describes the immediate value or cost associated with taking an
action while the system is in a particular state.

The reward function can be written as

$$
R(s,a)
$$

representing the immediate reward received If action $a$ is taken while the
system is in state $s$.

The objective of the agent is not necessarily to maximize the immediate reward
at every step, but to maximize the expected accumulated reward over time:

$$
\mathbb{E}
\left[
\sum_{t=0}^{\infty}
\gamma^t R(s_t,a_t)
\right]
$$

where $\gamma \in [0,1)$ is the discount factor.

For the standard crying baby POMDP , the reward rule is:
- being Hungry: **\(-10\)**
- taking Feed: additional **\(-5\)**
- taking Sing: additional **\(-0.5\)**
- taking Ignore: **no additional action cost**

This reward design encourages the agent to keep the baby full while avoiding unnecessary feeding.

In [7]:
def reward_function(state: State, action: Action):

    reward = 0

    if state == State.HUNGRY:
        reward -= 10

    if action == Action.FEED:
        reward -= 5
    elif action == Action.SING:
        reward -= 0.5

    return reward

### **Observation Space**

Observations are imperfect indicators of the true state. The agent uses these observations to update its belief about the underlying state.

$$
O = \{\text{Crying}, \text{Quiet}\}
$$

In [8]:
class Observation(Enum):
    Quiet = 0
    Crying = 1

### **Observation Model**

The observation model describes the probability of receiving a particular observation after the action the agent takes an action and transitioning to the new state $s'$. It is represented by:

$$
Z(o' \mid s', a) = P(o_{t+1}=o' \mid s_{t+1}=s', a_{t}=a)
$$

$$
s_t
\xrightarrow{a_t}
s_{t+1}
\xrightarrow{\text{observation}}
o_{t+1}
$$

If the caregiver **feeds** the baby and the baby's resulting state is **Hungry**,
there is a 0.8 probability of observing **crying** and a 0.2 probability of observing **quiet**.

$$
Z(\text{Crying} \mid \text{Hungry}, \text{Feed}) = 0.8
$$

$$
Z(\text{Quiet} \mid \text{Hungry}, \text{Feed}) = 0.2
$$

> Note: In this model, the state `Hungry` is unreachable after `Feed`,

If the caregiver **sings** to the baby and the baby's resulting state is **Hungry**,
there is a 0.9 probability of observing **crying**  and a 0.1 probability of observing **quiet**.

$$
Z(\text{Crying} \mid \text{Hungry}, \text{Sing}) = 0.9
$$

$$
Z(\text{Quiet} \mid \text{Hungry}, \text{Sing}) = 0.1
$$

If the caregiver **ignores** the baby and the baby's resulting state is **Hungry**,
there is a 0.8 probability of observing **crying**  and a 0.2 probability of observing **quiet**.

$$
Z(\text{Crying} \mid \text{Hungry}, \text{Ignore}) = 0.8
$$

$$
Z(\text{Quiet} \mid \text{Hungry}, \text{Ignore}) = 0.2
$$

If the caregiver **feeds** the baby and the baby's resulting state is **Full**,
there is a 0.1 probability of observing **crying**  and a 0.9 probability of observing **quiet**.

$$
Z(\text{Crying} \mid \text{Full}, \text{Feed}) = 0.1
$$

$$
Z(\text{Quiet} \mid \text{Full}, \text{Feed}) = 0.9
$$

If the caregiver **sings** to the baby and the baby's resulting state is **Full**,
there is a 0 probability of observing **crying**.

$$
Z(\text{Crying} \mid \text{Full}, \text{Sing}) = 0.0
$$

$$
Z(\text{Quiet} \mid \text{Full}, \text{Sing}) = 1.0
$$

If the caregiver **ignores** the baby and the baby's resulting state is **Full**,
there is a 0.1 probability of observing **crying**  and a 0.9 probability of observing **quiet**.

$$
Z(\text{Crying} \mid \text{Full}, \text{Ignore}) = 0.1
$$

$$
Z(\text{Quiet} \mid \text{Full}, \text{Ignore}) = 0.9
$$


In [9]:
def observation_probability(observation: Observation, state: State, action: Action):

    if state == State.HUNGRY:
        match action:

            case Action.FEED:
                return (0.8 if observation == Observation.Crying else 0.2)

            case Action.SING:
                return (0.9 if observation == Observation.Crying else 0.1)

            case Action.IGNORE:
                return (0.8 if observation == Observation.Crying else 0.2)

    else:
        match action:

            case Action.FEED:
                return (0.1 if observation == Observation.Crying else 0.9)

            case Action.SING:
                return (0.0 if observation == Observation.Crying else 1.0)

            case Action.IGNORE:
                return (0.1 if observation == Observation.Crying else 0.9)        

In [10]:
# Testing

print("State\t\tAction\t\t\tCrying\t\tQuiet")

for state in State:
    for action in Action:

        print(f"{state.name}\t\t{action.name}\t\t\t{observation_probability(Observation.Crying, state, action)}\t\t{observation_probability(Observation.Quiet, state, action)}")

State		Action			Crying		Quiet
HUNGRY		IGNORE			0.8		0.2
HUNGRY		SING			0.9		0.1
HUNGRY		FEED			0.8		0.2
FULL		IGNORE			0.1		0.9
FULL		SING			0.0		1.0
FULL		FEED			0.1		0.9


---

## **Belief**

As discussed, using past observations, the agent builds a **belief** of their underlying state. Mathematically speaking, it is a vector of a probability distribution over the true states.

$$
b = [b(s_1), b(s_2), ..., b(s_N)]
\\
s_1,...,s_N \in S
\\
b(s_i) \ge 0
\\
\sum_{i=1}^N b(s_i) = 1
$$

Belief is calculated based on the history of actions and observations, because only current observations might not be sufficient.

$$
h_t = (o_0, a_0, o_1, a_1, ..., a_{t-1}, o_t)
$$

$$
b_t(s) = P(s_t = s \mid h_t)
$$

By defining "belief", we turn a ***partially observable problem*** into a ***fully observable problem over beliefs***.

### **Belief Update**

the agent performs $a_t$, receives $o_{t+1}$.

$$
b_{t+1}(s')=P(s_{t+1}=s' \mid h_t, a_t, o_{t+1})
$$

#### **Step 1: Predict**

$$
\bar{b}_{t+1}(s') = \sum_{s} T(s' \mid s, a_t) b_t(s)
$$

> Given what i previously believed and the action I took, where do I think the world has moved?


#### **Step 2: Update**

$$
b_{t+1}(s_{t+1}) = P(s_{t+1} \mid h_t, a_t, o_{t+1})
$$

Based on Bayes' theorem:
$$
P(s_{t+1} \mid h_t, a_t, o_{t+1}) = \frac{P(o_{t+1} \mid h_t, a_t, s_{t+1})P(s_{t+1} \mid h_t, a_t)}{P(o_{t+1} \mid h_t, a_t)}
$$

**1**. Based on definitions:
$$
P(o_{t+1} \mid h_t, a_t, s_{t+1})
\\ = P(o_{t+1} \mid a_t, s_{t+1})
\\ = Z(o_{t+1} \mid a_t, s_{t+1})
$$

**2**. Trivially:
$$
P(s_{t+1} \mid h_t, a_t)
\\ = \sum_s T(s_{t+1} \mid s, a_t) P(s_t = s \mid h_t)
\\ = \sum_s T(s_{t+1} \mid s, a_t) b_t(s)
$$

**3**. And:
$$
P(o_{t+1} \mid h_t, a_t)
\\ = \sum_{s_{t+1}} P(o_{t+1}, s_{t+1} \mid h_t, a_t)
\\ = \sum_{s_{t+1}} P(o_{t+1} \mid s_{t+1}, h_t, a_t) P(s_{t+1} \mid h_t, a_t)
\\ = \sum_{s_{t+1}} Z(o_{t+1} \mid s_{t+1}, a_t) P(s_{t+1} \mid h_t, a_t)
\\ = \sum_{s_{t+1}} Z(o_{t+1} \mid s_{t+1}, a_t) \sum_{s_{t+1}} P(s_{t+1} \mid h_t, a_t)
\\ = \sum_{s_{t+1}} Z(o_{t+1} \mid s_{t+1}, a_t) \bar{b}_{t+1}(s_{t+1})
\\ 
\\ \rightarrow P(o_{t+1} \mid h_t, a_t) = \sum_{s_{t+1}} Z(o_{t+1} \mid s_{t+1}, a_t) \bar{b}_{t+1}(s_{t+1})
$$


**Based on 1, 2, 3:**

$$
b_{t+1}(s_{t+1}) = \frac{Z(o_{t+1} \mid a_t, s_{t+1}) \sum_s T(s_{t+1} \mid s, a_t) b_t(s)}{\sum_{s_{t+1}} Z(o_{t+1} \mid s_{t+1}, a_t) \bar{b}_{t+1}(s_{t+1})}
$$

$$
\rightarrow
b_{t+1}(s') = \eta Z(o_{t+1} \mid s', a_t) \bar{b}_{t+1} (s)
$$

where:

- $Z$ is to check the compatibility with the observation.
- $\bar{b}$ is the calculated prediction in step 1.
- $\eta$ is the normalization constant
$
=
\frac{1}{
\sum_{s'}
Z(o_{t+1}\mid s',a_t)\bar b_{t+1}(s')
}.
$

> Use the new observation to increase or decrease the probability of each predicted state depending on how compatible that state is with the observation.

In [11]:
import numpy as np
from numpy.typing import NDArray

In [12]:
def belief_predictor(current_belief: NDArray, next_state: State, current_action: Action):
    
    predicted_belief = 0
    for s in State:
        T = transition_probability(
        next_state=next_state,
        state=s,
        action=current_action
        )
        predicted_belief += T * current_belief[s.value]

    return predicted_belief

In [13]:
def update_belief(current_belief: NDArray, next_state: State, current_action: Action, observation: Observation):


    normalizer = 0
    for s in State:
        Z = observation_probability(
            observation=observation,
            state=s,
            action=current_action
            )
        predicted_belief = belief_predictor(current_belief=current_belief, next_state=s, current_action=current_action)
        normalizer += Z * predicted_belief

    eta = 1/normalizer

    Z = observation_probability(
        observation=observation,
        state=next_state,
        action=current_action
        )
    
    predicted_belief = belief_predictor(current_belief, next_state, current_action)
    
    new_belief = eta * Z * predicted_belief

    return new_belief

---

---

## **Environment**

The agent takes and action, moves to a next state sampled using T, gets rewarded, then the observation is sampled using Z.

In [ ]:
def step(state: State, action: Action):
    p_full = transition_probability(State.FULL, state, action)
    p_hungry = 1 - p_full
    possible_next_state = np.random.choice([State.FULL, State.HUNGRY], p=[p_full, p_hungry])

    reward = reward_function(state, action)

    p_crying = observation_probability(Observation.Crying, possible_next_state, action)
    p_quiet = 1 - p_crying
    possible_observation = np.random.choice([Observation.Crying, Observation.Quiet], p=[p_crying, p_quiet])

    return possible_next_state, reward, possible_observation

## **REFERENCES**

**[1]** M. Lauri, D. Hsu, and J. Pajarinen,<br>
        “Partially Observable Markov Decision Processes in Robotics: A Survey,”<br>
        *IEEE Transactions on Robotics*, vol. 39, no. 1, pp. 21–40, Feb. 2023.<br>